In [ ]:
################################################################################
# 3 x 4 ablation comparison of PINN and FD solution profiles
#
# Columns : Baseline PINN
#           Direct + Cons.
#           Log Density
#           Proposed PINN
#
# Rows    : Electrostatic potential psi(x)
#           Electron density n(x)
#           Mobile-ion density I(x)
#
# One representative seed is used consistently for all four formulations.
#
# FD   : solid blue line
# PINN : dashed orange line
################################################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# Working directory
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Root output directory
# -----------------------------------------------------------------------------
OUTPUT_ROOT = "results"

# -----------------------------------------------------------------------------
# Representative seed
# -----------------------------------------------------------------------------
REPRESENTATIVE_SEED = 42

# -----------------------------------------------------------------------------
# Ablation cases
# -----------------------------------------------------------------------------
cases = [
    {
        "title": "Direct",
        "directory": "ablation_direct",
        "summary": "ablation_direct_per_seed_summary.csv",
    },
    {
        "title": "Direct + Cons.",
        "directory": "ablation_direct_cons",
        "summary": "ablation_direct_cons_per_seed_summary.csv",
    },
    {
        "title": "Log-Density",
        "directory": "ablation_log_density",
        "summary": "ablation_log_density_per_seed_summary.csv",
    },
    {
        "title": "CA-PINN",
        "directory": "ablation_ca_pinn",
        "summary": "ablation_ca_pinn_per_seed_summary.csv",
    },
]

# -----------------------------------------------------------------------------
# Required result files
# -----------------------------------------------------------------------------
required_files = [
    "fd_psi.csv",
    "fd_n.csv",
    "fd_I.csv",
    "pinn_psi.csv",
    "pinn_n.csv",
    "pinn_I.csv",
]

# -----------------------------------------------------------------------------
# Helper function: return one representative output directory
# -----------------------------------------------------------------------------
def get_representative_output_directory(
    case,
    representative_seed,
):
    """
    Return the successful output directory corresponding to the selected
    representative seed.
    """
    summary_path = os.path.join(
        OUTPUT_ROOT,
        case["directory"],
        case["summary"],
    )

    if not os.path.isfile(summary_path):
        raise FileNotFoundError(
            f"Summary file not found:\n{summary_path}"
        )

    summary = pd.read_csv(summary_path)

    required_columns = [
        "status",
        "seed",
        "output_dir",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in summary.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {summary_path}: "
            f"{missing_columns}"
        )

    summary["seed"] = pd.to_numeric(
        summary["seed"],
        errors="coerce",
    )

    selected = summary.loc[
        summary["status"].eq("ok")
        & summary["seed"].eq(representative_seed)
    ].copy()

    if selected.empty:
        raise ValueError(
            f"No successful result for seed "
            f"{representative_seed} in:\n{summary_path}"
        )

    # Retain the latest matching row if duplicates exist.
    selected = selected.iloc[-1]

    output_directory = os.path.normpath(
        str(selected["output_dir"])
    )

    if not os.path.isdir(output_directory):
        raise FileNotFoundError(
            f"Output directory not found:\n"
            f"{output_directory}"
        )

    missing_files = [
        filename
        for filename in required_files
        if not os.path.isfile(
            os.path.join(
                output_directory,
                filename,
            )
        )
    ]

    if missing_files:
        raise FileNotFoundError(
            f"Missing files in:\n"
            f"{output_directory}\n\n"
            f"Missing: {missing_files}"
        )

    return output_directory

# -----------------------------------------------------------------------------
# Helper: read one profile
# -----------------------------------------------------------------------------
def read_profile(
    file_path,
    x_column,
    y_column,
):
    """
    Read one profile CSV and return sorted numerical arrays.
    """
    data = pd.read_csv(file_path)

    required_columns = [
        x_column,
        y_column,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {file_path}: "
            f"{missing_columns}"
        )

    x = pd.to_numeric(
        data[x_column],
        errors="coerce",
    ).to_numpy()

    y = pd.to_numeric(
        data[y_column],
        errors="coerce",
    ).to_numpy()

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        raise ValueError(
            f"Insufficient valid data in:\n{file_path}"
        )

    sorting_indices = np.argsort(x)

    return (
        x[sorting_indices],
        y[sorting_indices],
    )

# -----------------------------------------------------------------------------
# Locate the representative run for each formulation
# -----------------------------------------------------------------------------
case_directories = {}

for case in cases:

    output_directory = get_representative_output_directory(
        case=case,
        representative_seed=REPRESENTATIVE_SEED,
    )

    case_directories[case["title"]] = (
        output_directory
    )

    print(
        f"{case['title']}: "
        f"seed {REPRESENTATIVE_SEED}"
    )

print(
    "\nAll selected ablation folders and CSV files were found."
)

# -----------------------------------------------------------------------------
# Plot settings
# -----------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",

    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,

    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "axes.linewidth": 0.9,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.top": True,
    "ytick.right": True,

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "lines.linewidth": 2.0,

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
})

# -----------------------------------------------------------------------------
# Use the same colour coding as the Vi-sweep figure
# -----------------------------------------------------------------------------
FD_COLOR = "C0"
PINN_COLOR = "C1"

# -----------------------------------------------------------------------------
# Create the 3 x 4 figure
#
# Row 0: psi
# Row 1: n
# Row 2: I
#
# Column 0: Baseline
# Column 1: Direct + Cons.
# Column 2: Log Density
# Column 3: CA-PINN
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(
    nrows=3,
    ncols=4,
    figsize=(14.2, 8.2),
    sharex="col",
)

panel_labels = [
    ["(a)", "(b)", "(c)", "(d)"],
    ["(e)", "(f)", "(g)", "(h)"],
    ["(i)", "(j)", "(k)", "(l)"],
]

# -----------------------------------------------------------------------------
# Read and plot each ablation formulation
# -----------------------------------------------------------------------------
for j, case in enumerate(cases):

    case_title = case["title"]

    output_directory = case_directories[
        case_title
    ]

    # -------------------------------------------------------------------------
    # Load FD profiles
    # -------------------------------------------------------------------------
    fd_psi_x, fd_psi_y = read_profile(
        os.path.join(
            output_directory,
            "fd_psi.csv",
        ),
        x_column="x_nm",
        y_column="psi_V",
    )

    fd_n_x, fd_n_y = read_profile(
        os.path.join(
            output_directory,
            "fd_n.csv",
        ),
        x_column="x_nm",
        y_column="n_m3",
    )

    fd_I_x, fd_I_y = read_profile(
        os.path.join(
            output_directory,
            "fd_I.csv",
        ),
        x_column="x_nm",
        y_column="I_m3",
    )

    # -------------------------------------------------------------------------
    # Load PINN profiles
    # -------------------------------------------------------------------------
    pinn_psi_x, pinn_psi_y = read_profile(
        os.path.join(
            output_directory,
            "pinn_psi.csv",
        ),
        x_column="x_nm",
        y_column="psi_V",
    )

    pinn_n_x, pinn_n_y = read_profile(
        os.path.join(
            output_directory,
            "pinn_n.csv",
        ),
        x_column="x_nm",
        y_column="n_m3",
    )

    pinn_I_x, pinn_I_y = read_profile(
        os.path.join(
            output_directory,
            "pinn_I.csv",
        ),
        x_column="x_nm",
        y_column="I_m3",
    )

    # -------------------------------------------------------------------------
    # Row 1: Electrostatic potential
    # -------------------------------------------------------------------------
    ax[0, j].plot(
        fd_psi_x,
        fd_psi_y,
        color=FD_COLOR,
        linestyle="-",
        linewidth=2.2,
        label="FD",
        zorder=4,
    )

    ax[0, j].plot(
        pinn_psi_x,
        pinn_psi_y,
        color=PINN_COLOR,
        linestyle="--",
        linewidth=2.0,
        label="PINN",
        zorder=5,
    )

    ax[0, j].grid(
        True,
        which="major",
        linewidth=0.5,
        alpha=0.25,
    )

    ax[0, j].set_title(
        case_title,
        pad=22,
        fontweight="bold",
    )

    # -------------------------------------------------------------------------
    # Row 2: Electron density
    # -------------------------------------------------------------------------
    ax[1, j].semilogy(
        fd_n_x,
        fd_n_y,
        color=FD_COLOR,
        linestyle="-",
        linewidth=2.2,
        label="FD",
        zorder=4,
    )

    ax[1, j].semilogy(
        pinn_n_x,
        pinn_n_y,
        color=PINN_COLOR,
        linestyle="--",
        linewidth=2.0,
        label="PINN",
        zorder=5,
    )

    ax[1, j].grid(
        True,
        which="both",
        linewidth=0.5,
        alpha=0.25,
    )

    # -------------------------------------------------------------------------
    # Row 3: Mobile-ion density
    # -------------------------------------------------------------------------
    ax[2, j].semilogy(
        fd_I_x,
        fd_I_y,
        color=FD_COLOR,
        linestyle="-",
        linewidth=2.2,
        label="FD",
        zorder=4,
    )

    ax[2, j].semilogy(
        pinn_I_x,
        pinn_I_y,
        color=PINN_COLOR,
        linestyle="--",
        linewidth=2.0,
        label="PINN",
        zorder=5,
    )

    ax[2, j].grid(
        True,
        which="both",
        linewidth=0.5,
        alpha=0.25,
    )

    # -------------------------------------------------------------------------
    # Panel labels
    # -------------------------------------------------------------------------
    for i in range(3):

        ax[i, j].text(
            0.02,
            1.05,
            panel_labels[i][j],
            transform=ax[i, j].transAxes,
            ha="left",
            va="bottom",
            fontsize=11,
            fontweight="bold",
            clip_on=False,
        )

# -----------------------------------------------------------------------------
# Axis labels
#
# Show y-axis labels only in the first column.
# Show x-axis labels only in the bottom row.
# -----------------------------------------------------------------------------
ax[0, 0].set_ylabel(
    r"$\psi$ (V)"
)

ax[1, 0].set_ylabel(
    r"$n$ (m$^{-3}$)"
)

ax[2, 0].set_ylabel(
    r"$I$ (m$^{-3}$)"
)

for j in range(4):

    ax[2, j].set_xlabel(
        "Position (nm)"
    )

# -----------------------------------------------------------------------------
# Ensure a consistent spatial domain
# -----------------------------------------------------------------------------
for i in range(3):

    for j in range(4):

        ax[i, j].set_xlim(
            0,
            500,
        )

# -----------------------------------------------------------------------------
# Use common y-axis limits within each row
#
# This makes the magnitude of the failed and successful solutions directly
# comparable across all four formulations.
# -----------------------------------------------------------------------------
for row_index in range(3):

    row_limits = [
        ax[row_index, column_index].get_ylim()
        for column_index in range(4)
    ]

    row_minimum = min(
        limits[0]
        for limits in row_limits
    )

    row_maximum = max(
        limits[1]
        for limits in row_limits
    )

    for column_index in range(4):

        ax[row_index, column_index].set_ylim(
            row_minimum,
            row_maximum,
        )

# -----------------------------------------------------------------------------
# Common legend
# -----------------------------------------------------------------------------
handles, legend_labels = (
    ax[0, 0].get_legend_handles_labels()
)

fig.legend(
    handles,
    legend_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.015),
    ncol=2,
    frameon=False,
    handlelength=2.8,
    columnspacing=1.8,
)

# -----------------------------------------------------------------------------
# Layout adjustment
# -----------------------------------------------------------------------------
fig.subplots_adjust(
    left=0.075,
    right=0.99,
    bottom=0.085,
    top=0.88,
    wspace=0.28,
    hspace=0.28,
)

# -----------------------------------------------------------------------------
# Save the figure
# -----------------------------------------------------------------------------
figure_directory = os.path.join(
    OUTPUT_ROOT,
    "ablation_comparison",
)

os.makedirs(
    figure_directory,
    exist_ok=True,
)

pdf_path = os.path.join(
    figure_directory,
    "ablation_solution_profiles_3x4.pdf",
)

png_path = os.path.join(
    figure_directory,
    "ablation_solution_profiles_3x4.png",
)

plt.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight",
)

plt.savefig(
    png_path,
    format="png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()

print("\nSaved publication figures:")
print(pdf_path)
print(png_path)